In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

## Obteniendo los parámetros

In [0]:
%sql
CREATE WIDGET TEXT catalog_name DEFAULT 'smartdata_project';
CREATE WIDGET TEXT schema_source DEFAULT 'bronze';
CREATE WIDGET TEXT schema_sink DEFAULT 'silver';
CREATE WIDGET TEXT source_products DEFAULT 'bronze_products';
CREATE WIDGET TEXT sink_products DEFAULT 'silver_products';
CREATE WIDGET TEXT w_products DEFAULT '1900-01-01';

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema_source = dbutils.widgets.get('schema_source')
schema_sink = dbutils.widgets.get('schema_sink')
source_products = dbutils.widgets.get('source_products')
sink_products = dbutils.widgets.get('sink_products')
w_products = dbutils.widgets.get('w_products')

## Consiguiendo la tabla Silver

In [0]:
source_products_table = f'{catalog_name}.{schema_source}.{source_products}'
df_products = spark.sql(f"SELECT * FROM {source_products_table} WHERE updated_at > '{w_products}'")

### Evitamos los deduplicados

In [0]:
df_products_dep = (
  df_products
  .withColumn('rn', row_number().over(Window.partitionBy(col('product_id')).orderBy(col('updated_at').desc())))
).filter(col('rn') == 1).drop('rn')

### Validamos product_id y price

In [0]:
df_products_invalid = df_products_dep.filter(
    col('product_id').isNull() |
    col('price').isNull() |
    (col('price') <= 0)
)

invalid_count = df_products_invalid.count()

if invalid_count > 0:
    print(f"⚠️ {invalid_count} registros inválidos:")

    df_quarantine = (
        df_products_dep
        .filter(col('price') <= 0)
        .withColumn('rejection_reason', when(col('price') == 0, "price is zero")
            .when(col('price') < 0,  "price is negative")
            .otherwise("price is null")
        ).withColumn("quarantined_at", current_timestamp())
    )

    df_quarantine.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog_name}.{schema_sink}.quarantine_products")

    df_products_dep = df_products_dep.filter(
        ~(
            col('product_id').isNull() |
            col('price').isNull() |
            (col('price') <= 0)
        )   
    )

else:
    print("✅ Todos los registros son válidos")

In [0]:
df_silver_products = (
    df_products_dep
    .filter(col('price') > 0)
    .filter(col('product_id').isNotNull())

    .withColumn("price", round(col('price'), 2))

    .withColumn("category_name",   initcap(trim(col('category_name'))))
    .withColumn("department_name", initcap(trim(col('department_name'))))

    .drop("_ingestion_timestamp")
    .withColumn("_ingestion_timestamp", current_timestamp())
)

In [0]:
target_person_table = f'{catalog_name}.{schema_sink}.{sink_products}'
if spark.catalog.tableExists(target_person_table):

    source_table = DeltaTable.forName(spark, target_person_table)

    (
        source_table.alias("target")
        .merge(
            df_silver_products.alias("source"),
            f"target.product_id = source.product_id"
        )
        # Registro existe y cambió algo → actualizar
        .whenMatchedUpdateAll()
        # Registro nuevo → insertar
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        df_silver_products.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .clusterBy('product_id','updated_at','price')  # Liquid Clustering
        .saveAsTable(target_person_table)
    )